# Deliverable 3
### By Kevin Honningsvåg and Herman Nordstrand

### GitHub: https://github.com/KevinHonningsvaag/inf_201_deliverable_3

### Imports

In [124]:
from __future__ import annotations

import random
from typing import override

## Task 1

In [125]:
class Character:
    """Represents a character in the RPG"""

    def __init__(
        self,
        name: str,
        weapon: Weapons | None = None,
        hp_max: int = 100,
        base_damage: int = 10,
    ) -> None:
        """
        Initialise the Character

        Args:
            name (str): The name of the character.
            hp_max (int): Max hit points (default = 100)
            damage (int): Damage delt when character attacks (default = 10)
        """
        self.name = name
        self._hp = hp_max  # Note: _hp indicated private attribute
        self.hp_max = hp_max
        if weapon is None:
            self.damage = base_damage
        else:
            self.damage = weapon.damage

    def attack(self, target: Character) -> None:
        """Attack target with a random amount of damage."""
        damage = Dice(self.damage).roll()
        target.take_damage(damage)

    def take_damage(self, damage: int) -> None:
        """Reduce HP by the incoming damage."""
        if damage < 0:
            raise ValueError("Damage cannot be negative")

        self._hp = max(self._hp - damage, 0)

    def heal(self, heal: int) -> None:
        """Character recovers hitpoints from healing"""
        self._hp += heal
        self._hp = min(self._hp, self.hp_max)  # Target hp cannot end up higher than hp_max

    def introduce(self) -> None:
        """Creates string that introduces character"""
        print(f"Hi, my name is {self.name}!")

    def __repr__(self) -> str:
        """Representation of character object"""
        return f"<Character name={self.name} hp={self.hp} damage={self.damage}>)"

    def __str__(self) -> str:
        """String representation of character object"""
        return f"Character: {self.name} with {self.hp} HP and {self.damage} damage"

    @property
    def hp(self) -> int:
        """Public read-only property to inspect HP without allowing direct, unsafe edits."""
        return self._hp

    @property
    def is_alive(self) -> bool:
        """Returns if the character is alive or not"""
        return self._hp > 0

## Task 2

Dice class
So the damage is variable and not fixed

In [126]:
class Dice:
    def __init__(self, sides: int = 6) -> None:
        """
        Initialise the dice

        Args:
            sides (int) : how many sides does the dice have
        """
        if sides < 2:
            raise ValueError("Dice must have atleast 2 sides")
        self.sides = sides

    def roll(self) -> int:
        """Rolls the dice and makes it land on a random side"""
        return random.randint(1, self.sides)

    def __repr__(self) -> str:
        """Representation of dice object"""
        return f"Dice(sides = {self.sides})"

    def __call__(self) -> int:
        """Make the diceroll happen"""
        return self.roll()

Showing the 

In [127]:
d6 = Dice()
d20 = Dice(20)

print(f"Default dice: {d6}")
print(f"20-sided dice: {d20}")

print(f"Roll with d6: {d6.roll()}")
print(f"Roll with d20: {d20.roll()}")
print(f"Roll using __call__: {d20()}")

Default dice: Dice(sides = 6)
20-sided dice: Dice(sides = 20)
Roll with d6: 6
Roll with d20: 12
Roll using __call__: 5


## Task 3
Creating items

In [128]:
class Weapons:
    def __init__(self, name: str, damage: int) -> None:
        self.name = name
        self.damage = damage

## Task 4
Charcter specialization
Sub-classes: Mage & Warrior

In [129]:
class Warrior(Character):
    """Represents a warrior character."""

    def __init__(
        self,
        name: str,
        weapon: Weapons | None = None,
        # 150 hp is too much, but warrior should have more hp, but in this system it makes the warrior too strong, so gave both 100 hp
        hp_max: int = 100,
        base_damage: int = 10,
        armor: int = 5,
    ) -> None:
        super().__init__(name=name, weapon=weapon, hp_max=hp_max, base_damage=base_damage)
        self.armor: int = armor
        if armor < 0:
            raise ValueError("armor need to be positive")

    @override
    def take_damage(self, damage: int) -> None:
        """Reduce incoming damage by armor."""
        if damage < 0:
            raise ValueError("cannot take negative damage")

        reduced_damage = max(damage - self.armor, 0)
        super().take_damage(reduced_damage)


class Mage(Character):
    """Represents a mage character."""

    def __init__(
        self,
        name: str,
        weapon: Weapons | None = None,
        hp_max: int = 100,
        base_damage: int = 10,
        mana: int = 30,
        mana_max: int = 30,
    ) -> None:
        super().__init__(name, weapon, hp_max, base_damage)
        self.base_damage: int = base_damage
        self.mana: int = mana
        self.mana_max: int = mana_max

        if not 0 <= mana <= mana_max:
            raise ValueError("mana requirement: 0 <= mana <= mana_max")

    @override
    def attack(self, target: Character) -> None:
        """Use magic if enough mana is available, otherwise attack normally."""
        if self.mana >= 10:
            self.mana -= 10
            target.take_damage(Dice(self.base_damage).roll() * 2)  # Mana attack deals double damage

        else:
            super().attack(target)
            self.mana = min(self.mana + 5, self.mana_max)  # Regenerates mana when using base attack

## Task 5
Running battles

In [130]:
class Battle:
    """Represents a battle between two characters"""

    def __init__(self, hero: Character, enemy: Character) -> None:
        """
        choose two characters to battle each other
        """
        self.hero = hero
        self.enemy = enemy

    def play_round(self) -> None:
        """simulates one round of battle between hero and enemy"""

        if self.hero.is_alive and self.enemy.is_alive:
            enemy_hp_before = self.enemy.hp
            self.hero.attack(self.enemy)

            print(f"{self.hero.name} attacks {self.enemy.name}!")

            damage_dealt = enemy_hp_before - self.enemy.hp

            print(f"{self.hero.name} deals {damage_dealt} damage to {self.enemy.name}!")
            print(f"{self.enemy.name} has {self.enemy.hp} HP remaining.")

            if self.enemy.is_alive:
                hero_hp_before = self.hero.hp
                self.enemy.attack(self.hero)
                damage_dealt = hero_hp_before - self.hero.hp

                print(f"{self.enemy.name} deals {damage_dealt} damage to {self.hero.name}!")
                print(f"{self.hero.name} has {self.hero.hp} HP remaining.")

    def simulate_battle(self) -> None:
        """Simulate a battle until one character dies."""

        round_number = 1

        print(f"\nBattle: {self.hero.name} vs {self.enemy.name}")

        while self.hero.is_alive and self.enemy.is_alive:
            print(f"\n========== Round {round_number} ==========")

            self.play_round()
            round_number += 1

        if self.hero.is_alive:
            print(
                f"\n{self.hero.name} wins the battle, with {self.hero.hp} HP remaining after {round_number - 1} rounds!"
            )
        elif self.enemy.is_alive:
            print(
                f"\n{self.enemy.name} wins the battle, with {self.enemy.hp} HP remaining after {round_number - 1} rounds!"
            )
        else:
            print("\nNo winner!")

# Results

without weapons

In [131]:
hero = Warrior("Isildur")
enemy = Mage("Sauron")

battle = Battle(hero, enemy)
battle.simulate_battle()


Battle: Isildur vs Sauron

========== Round 1 ==========
Isildur attacks Sauron!
Isildur deals 10 damage to Sauron!
Sauron has 90 HP remaining.
Sauron deals 0 damage to Isildur!
Isildur has 100 HP remaining.

========== Round 2 ==========
Isildur attacks Sauron!
Isildur deals 4 damage to Sauron!
Sauron has 86 HP remaining.
Sauron deals 11 damage to Isildur!
Isildur has 89 HP remaining.

========== Round 3 ==========
Isildur attacks Sauron!
Isildur deals 10 damage to Sauron!
Sauron has 76 HP remaining.
Sauron deals 0 damage to Isildur!
Isildur has 89 HP remaining.

========== Round 4 ==========
Isildur attacks Sauron!
Isildur deals 10 damage to Sauron!
Sauron has 66 HP remaining.
Sauron deals 0 damage to Isildur!
Isildur has 89 HP remaining.

========== Round 5 ==========
Isildur attacks Sauron!
Isildur deals 6 damage to Sauron!
Sauron has 60 HP remaining.
Sauron deals 2 damage to Isildur!
Isildur has 87 HP remaining.

========== Round 6 ==========
Isildur attacks Sauron!
Isildur deals

with weapons

In [132]:
sword = Weapons("Sword", 15)
staff = Weapons("Staff", 12)

hero = Warrior("Elendil", weapon=sword)
enemy = Mage("Sauron", weapon=staff)

battle = Battle(hero, enemy)
battle.simulate_battle()


Battle: Elendil vs Sauron

========== Round 1 ==========
Elendil attacks Sauron!
Elendil deals 11 damage to Sauron!
Sauron has 89 HP remaining.
Sauron deals 9 damage to Elendil!
Elendil has 91 HP remaining.

========== Round 2 ==========
Elendil attacks Sauron!
Elendil deals 2 damage to Sauron!
Sauron has 87 HP remaining.
Sauron deals 0 damage to Elendil!
Elendil has 91 HP remaining.

========== Round 3 ==========
Elendil attacks Sauron!
Elendil deals 2 damage to Sauron!
Sauron has 85 HP remaining.
Sauron deals 7 damage to Elendil!
Elendil has 84 HP remaining.

========== Round 4 ==========
Elendil attacks Sauron!
Elendil deals 13 damage to Sauron!
Sauron has 72 HP remaining.
Sauron deals 1 damage to Elendil!
Elendil has 83 HP remaining.

========== Round 5 ==========
Elendil attacks Sauron!
Elendil deals 2 damage to Sauron!
Sauron has 70 HP remaining.
Sauron deals 0 damage to Elendil!
Elendil has 83 HP remaining.

========== Round 6 ==========
Elendil attacks Sauron!
Elendil deals 9 

## Discussing code

We continued working on the Character-class form the lecture. ANd the character-class is the foundation of this rpg simulator. The class contains traits such as name, healt(HP), maxHP and damage. It also contains other methods that allows the characters to attack eachother and recieve damage to their HP.
We used encapsulation so that we rresitrict the access to some attributes. By using _hp we signal that HP should not be changed from the outside. 

The way the code is done makes it so that we can reuse a lot of code by inheriting traits. As both mage and warrior have different traits, they also inherits traits from Character.

The way we have setup the mage and the warrior class now is a major advantage to the warrior in future iterations the magic attack should propably ignore the armor to some degree, depending on different types of armor to expand what type of armor and types of protection, so some classes should be better against eachother, like the pokemon system. Where if we were to expand you could pick a team. would of course have to expand on classes aswell

By having an own dice class for making the damage random. This makes the battles less predictable, but that depends on the differences of the classes aswell.
A limitation of the RPG simulation is that it is quite simple, which results in really unambigous gameplay, as the characters always figt in the same order, and the attacks are the same.

If this was to be an actual game we would definitly expand the weapons section where you for instance could have a wizard staff that gave a max mana boost aswell as a boost in mana recovered per round. So some items give good buffs, but have some negative effects aswell. 
but there should also be some healing items, and more elements to potentially turn the tide of the battle.

With the current code, i have never seen warrior lose a battle. which should be improved for further development by the things mentioned above.



## GenAi discussion

As we tried something new for us with pytest and ruff formatting, it was very helpful to use AI for bugfixing, where we would get errormessages and no idea what they meant or how to fix them. AI helped massively there which ups the effiecency of the project. Though we yet again strive to use AI as little as possible for our own sake and to get as much as possible out of every single assignment.